# Phần 3: chạy Dense (E5) trên pool Test và Train (Kaggle GPU)

**Notebook này là bản ghi lại lần chạy thật trên Kaggle (GPU T4), dùng để xem lại. Không có ý định chạy lại trên máy cá nhân** (cần GPU, dataset pool và code đã tải lên Kaggle).

Đây là **phần 3 trong 7 phần** của Pipeline 1: (1) dựng pool, (2) chạy BM25, **(3) chạy Dense**, (4) train, (5) val, (6) test, (7) kết quả.

**Việc notebook này làm:** dùng mô hình `intfloat/multilingual-e5-small` (đa ngôn ngữ, nhẹ) để biến câu hỏi và từng đoạn văn thành vector, rồi xếp hạng tài liệu theo độ giống nhau. Chạy với đúng 2 cách cắt đoạn như BM25 để so sánh được: `fixed_window_dense` (cửa sổ cố định 256 từ, bước 128) và `structure_dense` (đoạn theo cấu trúc).

**Vì sao bật Internet:** chỉ để tải mô hình E5 một lần. Phần chạy thực nghiệm dùng bản đã tải về (`local_files_only`).

**Các bước:**  
1. copy code, vá; 
2. copy pool, duyệt governance;
3. tạo config; 
4. tải mô hình và đo tốc độ thật; bật fp16;
5. chạy Test; (5b) bật 2 GPU; 
6. chạy Train; 
7. đọc kết quả, nén zip; 
8. tách theo loại đạo văn và bootstrap cặp; phụ lục xuất đáp án để phân tích ngoài Kaggle.

### Bước 1 — Copy code, vá 2 chỗ

Dataset code trên Kaggle vẫn là bản cũ nên vá thủ công:
1. `read_jsonl`: tách dòng bằng `split("\n")` thay vì `splitlines()` (lỗi ký tự `\x85`).
2. `runner.py`: thêm phương pháp `fixed_window_dense`.

Mỗi chỗ vá đều kiểm tra "đúng 1 vị trí khớp", sai là báo lỗi ngay thay vì vá bừa.

In [2]:
import glob, shutil, os

hits = glob.glob('/kaggle/input/**/requirements-research.txt', recursive=True)
assert hits, 'Không thấy requirements-research.txt: kiểm tra đã gắn dataset code chưa'
RESEARCH_SRC = os.path.dirname(hits[0])
os.makedirs('/kaggle/working/repo', exist_ok=True)
shutil.copytree(RESEARCH_SRC, '/kaggle/working/repo/research', dirs_exist_ok=True)
!pip install -q -r /kaggle/working/repo/research/requirements-research.txt

def patch(path, pairs):
    s = open(path, encoding="utf-8").read()
    for old, new in pairs:
        assert s.count(old) == 1, f"{path}: tìm thấy {s.count(old)} chỗ, cần đúng 1 :: {old[:50]}"
        s = s.replace(old, new)
    open(path, "w", encoding="utf-8").write(s)

R = "/kaggle/working/repo/research/offline_retrieval"

patch(f"{R}/common.py", [(
    'enumerate(path.read_text(encoding="utf-8").splitlines(), 1)',
    'enumerate(path.read_text(encoding="utf-8").split("\\n"), 1)')])

patch(f"{R}/runner.py", [
 ('''ALLOWED_METHODS = {
    "dense_cross_encoder",
    "fixed_window_bm25",''',
  '''ALLOWED_METHODS = {
    "dense_cross_encoder",
    "fixed_window_bm25",
    "fixed_window_dense",'''),
 ('''    fixed_windows: list[dict[str, Any]] = []
    fixed_bm25: dict[str, list[dict[str, Any]]] = {}
    if needs_fixed_bm25:
        stage_start = time.perf_counter()
        fixed_windows = build_fixed_windows(
            loaded["documents"],
            size_tokens=int(config["fixed_window"]["size_tokens"]),
            stride_tokens=int(config["fixed_window"]["stride_tokens"]),
        )
        stage_timings["fixed_window_build"] = time.perf_counter() - stage_start
        stage_start = time.perf_counter()''',
  '''    needs_fixed_dense = "fixed_window_dense" in selected_methods
    fixed_windows: list[dict[str, Any]] = []
    fixed_bm25: dict[str, list[dict[str, Any]]] = {}
    if needs_fixed_bm25 or needs_fixed_dense:
        stage_start = time.perf_counter()
        fixed_windows = build_fixed_windows(
            loaded["documents"],
            size_tokens=int(config["fixed_window"]["size_tokens"]),
            stride_tokens=int(config["fixed_window"]["stride_tokens"]),
        )
        stage_timings["fixed_window_build"] = time.perf_counter() - stage_start
    if needs_fixed_bm25:
        stage_start = time.perf_counter()'''),
 ('''    if "structure_hybrid_rrf" in selected_methods:
        stage_start = time.perf_counter()
        rrf_key =''',
  '''    if needs_fixed_dense:
        if encoder is None:
            encoder = _make_encoder(config, loaded["config_path"])
        stage_start = time.perf_counter()
        fixed_dense = rank_dense(
            fixed_windows,
            loaded["queries"],
            encoder=encoder,
            top_k=candidate_depth,
            group_by_document=target_scope == "document",
        )
        stage_timings["fixed_window_dense"] = time.perf_counter() - stage_start
        rankings["fixed_window_dense"] = {
            query_id: candidates[:depth]
            for query_id, candidates in fixed_dense.items()
        }

    if "structure_hybrid_rrf" in selected_methods:
        stage_start = time.perf_counter()
        rrf_key ='''),
])
print("Đã vá xong common.py và runner.py")

Đã vá xong common.py và runner.py


### Bước 2 — Copy pool Test/Train sang vùng ghi được, duyệt governance

**Vì sao phải copy:** `/kaggle/input` (nơi dataset được gắn vào) chỉ đọc, mà bước duyệt governance cần ghi vào `dataset-manifest.json`. Nên copy 2 pool đã dựng sẵn (`pan-pc11-test-v1`, `pan-pc11-train-v1`) sang `/kaggle/working/data`, là vùng ghi được. Việc này chỉ copy dữ liệu đã có, không dựng lại pool.

**Vì sao phải duyệt governance:** runner từ chối chạy nếu `dataset-manifest.json` chưa ghi `governance = APPROVED` (lỗi "Dataset is not APPROVED"). Đây là thiết kế cố ý với dữ liệu bên thứ ba. PAN-PC-11 là corpus công khai cho nghiên cứu theo điều khoản Zenodo, nên tác giả tự duyệt và ghi tên người duyệt, ngày duyệt vào manifest để truy vết.

**Kết quả đúng:** in ra `TEST_DIR`, `TRAIN_DIR`, và dòng `governance: APPROVED` cho cả hai pool. Hai biến `TEST_DIR`, `TRAIN_DIR` được các bước sau dùng, nên ô này phải chạy trước Bước 4.

*Nếu phiên bị restart (ví dụ đổi GPU), mất các biến này và phải chạy lại ô này.*

In [10]:
import glob, json, pathlib, os, shutil

def copy_pool_dir(folder_name):
    hits = glob.glob(f'/kaggle/input/**/{folder_name}', recursive=True)
    assert hits, f'Không tìm thấy {folder_name}: kiểm tra đã gắn dataset pool chưa'
    src = hits[0]
    dst = f"/kaggle/working/data/{folder_name}"
    if not os.path.exists(dst):
        print("Đang copy", folder_name, "sang vùng ghi được...")
        shutil.copytree(src, dst)
    return dst

TEST_DIR  = copy_pool_dir("pan-pc11-test-v1")
TRAIN_DIR = copy_pool_dir("pan-pc11-train-v1")
print("TEST_DIR =", TEST_DIR)
print("TRAIN_DIR =", TRAIN_DIR)

def approve_governance(dataset_dir: str, approved_at: str):
    manifest_path = pathlib.Path(dataset_dir) / "dataset-manifest.json"
    manifest = json.loads(manifest_path.read_text())
    manifest["governance"].update({
        "status": "APPROVED",
        "contains_pii": False,
        "retention": "Ephemeral Kaggle run; outputs giu lai lam bang chung thuc nghiem cho KLTN",
        "approver": "Vo Thi Kim Yen (tac gia KLTN, tu duyet du lieu cong khai PAN-PC-11 theo dieu khoan Zenodo)",
        "approved_at": approved_at,
    })
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
    print(dataset_dir, "-> governance:", manifest["governance"]["status"])

approve_governance(TEST_DIR, "2026-11-19")
approve_governance(TRAIN_DIR, "2026-11-20")

Đang copy pan-pc11-test-v1 sang vùng ghi được...
Đang copy pan-pc11-train-v1 sang vùng ghi được...
TEST_DIR = /kaggle/working/data/pan-pc11-test-v1
TRAIN_DIR = /kaggle/working/data/pan-pc11-train-v1
/kaggle/working/data/pan-pc11-test-v1 -> governance: APPROVED
/kaggle/working/data/pan-pc11-train-v1 -> governance: APPROVED


### Bước 3 — Tạo file config cho thực nghiệm Dense

Dataset code trên Kaggle chưa có file config Dense mới, nên tạo ngay trong notebook (nội dung giống hệt file trong repo `pan-pc-11-dense-chunking-pilot-v1.json`). Hai phương pháp cần chạy: `fixed_window_dense` và `structure_dense`. Model E5-small đã khoá đúng commit, chạy trên GPU, batch 128.

In [11]:
import json
CONFIG_PATH = "/kaggle/working/dense_chunking_config.json"
dense_config = {
  "schema_version": 1,
  "experiment_id": "pan-pc-11-dense-chunking-pilot-v1",
  "protocol": {
    "dataset_role": "CORE_SOURCE_RETRIEVAL_DIAGNOSTIC",
    "query_unit": "annotated_suspicious_span_window",
    "target_scope": "source_document",
    "corpus_scope": "DETERMINISTIC_RESOURCE_BOUNDED_PILOT",
    "comparison": "structure_dense_vs_fixed_window_dense_matched_chunking_with_bm25",
    "full_corpus_result": False,
    "official_pan_result": False
  },
  "seed": 20260923,
  "methods": ["fixed_window_dense", "structure_dense"],
  "candidate_retrieval_depth": 200,
  "retrieval_depth": 100,
  "metric_cutoffs": [1, 5, 10, 20, 100],
  "fixed_window": {"size_tokens": 256, "stride_tokens": 128},
  "bm25": {"k1": 1.2, "b": 0.75},
  "dense": {
    "backend": "sentence_transformers",
    "model": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "revision_source": "https://huggingface.co/intfloat/multilingual-e5-small/commit/614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "device": "cuda",
    "batch_size": 128,
    "query_prefix": "query: ",
    "document_prefix": "passage: "
  },
  "rrf": {"k": 60},
  "metrics": {"target_scope": "document", "gold_overlap_threshold": 0.5}
}
json.dump(dense_config, open(CONFIG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã ghi", CONFIG_PATH)

Đã ghi /kaggle/working/dense_chunking_config.json


### Bước 4 — Tải mô hình E5 và đo tốc độ thật trên một mẫu nhỏ

**Làm gì:** tải mô hình một lần (cần bật Internet), rồi mã hoá thử 3.000 cửa sổ để đo tốc độ thật (cửa sổ mỗi giây) và ước lượng thời gian cho cả pool.

**Vì sao:** các ước lượng ban đầu chỉ là phỏng đoán. Đo trên mẫu nhỏ giúp biết trước thời gian chạy, tránh đốt giờ GPU khi tốc độ chậm hơn dự kiến.

**Kết quả đúng:** in ra tốc độ, ví dụ "118 cửa sổ/giây" (độ chính xác fp32) và thời gian ước lượng cho Test.

In [12]:
import sys, time, json
from huggingface_hub import snapshot_download

MODEL, REV = "intfloat/multilingual-e5-small", "614241f622f53c4eeff9890bdc4f31cfecc418b3"
snapshot_download(MODEL, revision=REV)

REPO = "/kaggle/working/repo"
sys.path.insert(0, REPO)
from research.offline_retrieval.dense import SentenceTransformerEncoder
from research.offline_retrieval.fixed_window import build_fixed_windows
from research.offline_retrieval.runner import validate_inputs

loaded = validate_inputs(f"{TEST_DIR}/dataset-manifest.json", CONFIG_PATH)
wins = build_fixed_windows(loaded["documents"][:300], size_tokens=256, stride_tokens=128)[:3000]

enc = SentenceTransformerEncoder(model=MODEL, revision=REV, device="cuda", batch_size=128,
                                 query_prefix="query: ", document_prefix="passage: ")
enc.set_mode("document")
enc.encode([w["text"] for w in wins[:256]])           # khởi động GPU, không tính giờ
t0 = time.perf_counter()
enc.encode([w["text"] for w in wins])
dt = time.perf_counter() - t0
rate = len(wins) / dt
print(f"{len(wins)} cửa sổ trong {dt:.1f}s = {rate:.0f} cửa sổ/giây")
for name, n in [("Test: 600.890 cửa sổ", 600_890), ("Test: 375.876 đoạn structure", 375_876)]:
    print(f"  ước lượng {name}: {n/rate/60:.0f} phút")

3000 cửa sổ trong 25.5s = 118 cửa sổ/giây
  ước lượng Test: 600.890 cửa sổ: 85 phút
  ước lượng Test: 375.876 đoạn structure: 53 phút


### Bước 4b — Bật chế độ fp16 cho Dense

Đo trên T4: fp32 chạy 118 cửa sổ/giây, fp16 chạy 334 cửa sổ/giây (nhanh gấp 2,8 lần). Độ giống vector giữa fp32 và fp16: trung bình ≈ 1,000, thấp nhất 0,9998, nên xếp hạng gần như không đổi. Dùng fp16 cho **mọi** lần chạy Dense (cả 2 cách chunking, cả Test và Train) để so sánh công bằng, và ghi rõ trong phần phương pháp của bài.

Dataset code trên Kaggle chưa có tuỳ chọn `dtype`, nên vá thêm 2 file (`dense.py`, `runner.py`) và đặt `"dtype": "float16"` trong config.

In [14]:
def patch(path, pairs):
    s = open(path, encoding="utf-8").read()
    for old, new in pairs:
        assert s.count(old) == 1, f"{path}: tìm thấy {s.count(old)} chỗ, cần đúng 1 :: {old[:50]}"
        s = s.replace(old, new)
    open(path, "w", encoding="utf-8").write(s)

R = "/kaggle/working/repo/research/offline_retrieval"

patch(f"{R}/dense.py", [
 ('''        query_prefix: str = "",
        document_prefix: str = "",
    ) -> None:
        if not model or not revision''',
  '''        query_prefix: str = "",
        document_prefix: str = "",
        dtype: str = "float32",
    ) -> None:
        if dtype not in {"float32", "float16"}:
            raise ValueError("Dense dtype must be float32 or float16")
        if dtype == "float16" and not str(device).startswith("cuda"):
            raise ValueError("Dense dtype float16 requires a CUDA device")
        if not model or not revision'''),
 ('''        self._model = SentenceTransformer(model, **kwargs)
        self._batch_size = batch_size''',
  '''        self._model = SentenceTransformer(model, **kwargs)
        if dtype == "float16":
            self._model.half()
        self._batch_size = batch_size'''),
 ('''            "batch_size": batch_size,
            "local_files_only": True,''',
  '''            "batch_size": batch_size,
            "dtype": dtype,
            "local_files_only": True,'''),
])
patch(f"{R}/runner.py", [
 ('''        document_prefix=dense.get("document_prefix", ""),
    )
    encoder.metadata["model"]''',
  '''        document_prefix=dense.get("document_prefix", ""),
        dtype=dense.get("dtype", "float32"),
    )
    encoder.metadata["model"]'''),
])

dense_config["dense"]["dtype"] = "float16"
json.dump(dense_config, open(CONFIG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã vá dense.py, runner.py; config dtype =", json.load(open(CONFIG_PATH))["dense"]["dtype"])

Đã vá dense.py, runner.py; config dtype = float16


### Bước 5 — Chạy Dense (2 cách chunking) trên Test

**Việc ô này làm:** gọi runner chạy 2 phương pháp trên pool Test (300 parent, 2.606 câu hỏi):
- `fixed_window_dense`: cắt văn bản thành cửa sổ cố định 256 từ (bước 128), mã hoá từng cửa sổ bằng E5 (600.890 cửa sổ).
- `structure_dense`: mã hoá từng đoạn theo cấu trúc văn bản (375.876 đoạn).

Mỗi câu hỏi và mỗi đoạn được biến thành một vector 384 chiều; độ giống nhau đo bằng cosine. Với mỗi câu hỏi, lấy điểm cao nhất trong số các đoạn của từng tài liệu làm điểm của tài liệu đó, rồi xếp hạng tài liệu và giữ top 100. Cách gộp theo tài liệu này giống hệt BM25, nên so sánh được trực tiếp.

**Cấu hình:** model `intfloat/multilingual-e5-small` (khoá đúng commit), GPU Tesla T4, độ chính xác **fp16**, batch 128. Đã kiểm tra fp16 so với fp32: độ giống vector trung bình ≈ 1,000, thấp nhất 0,9998, tốc độ nhanh gấp 2,8 lần (334 so với 118 cửa sổ/giây).

**Thời gian:** ước lượng ban đầu khoảng 50 phút; thực tế 60,1 phút (khoảng 38 phút cho cửa sổ cố định, 20 phút cho đoạn structure). Ô này **im lặng** đến khi xong vì runner không in tiến độ. Để biết nó vẫn chạy, nhìn bảng tài nguyên bên phải: GPU phải ở mức cao.

**Kết quả đúng:** in `test: rc=0 | ... phút`. Kết quả ghi vào `/kaggle/working/results/test/dense_chunking/` (`predictions.jsonl`, `metrics.json`, `run_manifest.json`).

*Chạy Test trước; chỉ chạy Train (Bước 6) sau khi Test ra `rc=0` và Recall hợp lý, để không đốt giờ GPU khi có lỗi.*

In [15]:
import subprocess, pathlib, time, os
OUT_ROOT = "/kaggle/working/results"

def run_dense(dataset_dir, split_name):
    out_dir = f"{OUT_ROOT}/{split_name}/dense_chunking"
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
    cmd = ["python", "-m", "research.offline_retrieval", "run",
           "--dataset-manifest", f"{dataset_dir}/dataset-manifest.json",
           "--config", CONFIG_PATH, "--output-dir", out_dir]
    t0 = time.perf_counter()
    print("Đang chạy Dense:", split_name, "...")
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("LỖI:", r.stderr[-3000:])
    print(f"{split_name}: rc={r.returncode} | {(time.perf_counter()-t0)/60:.1f} phút")
    return r.returncode

rc_test = run_dense(TEST_DIR, "test")

Đang chạy Dense: test ...
/kaggle/working/results/test/dense_chunking/run_manifest.json

test: rc=0 | 60.1 phút


In [16]:
import os, json
OUT_ROOT = "/kaggle/working/results"
for split in ["test"]:
    d = f"{OUT_ROOT}/{split}/dense_chunking"
    m = json.load(open(f"{d}/metrics.json"))
    print("=====", split, "=====")
    for method, v in m.items():
        if isinstance(v, dict) and "parent_macro" in v:
            row = {k: round(v["parent_macro"][k]["recall"], 4) for k in ["1", "10", "100"]}
            print(f"{method:20s} parent recall@k = {row}")
    rm = json.load(open(f"{d}/run_manifest.json"))
    print("timing_seconds:", rm.get("timing_seconds"))
    print("dense_encoder:", rm.get("dense_encoder"))

===== test =====
fixed_window_dense   parent recall@k = {'1': 0.672, '10': 0.7746, '100': 0.8895}
structure_dense      parent recall@k = {'1': 0.6081, '10': 0.7378, '100': 0.8543}
timing_seconds: {'evaluation': 0.4151437629998327, 'fixed_window_build': 51.553120374999935, 'fixed_window_dense': 2296.6692107220006, 'structure_dense': 1200.683768072}
dense_encoder: {'backend': 'sentence_transformers', 'batch_size': 128, 'device': 'cuda', 'document_prefix': 'passage: ', 'dtype': 'float16', 'local_files_only': True, 'model': 'intfloat/multilingual-e5-small', 'query_prefix': 'query: ', 'revision': '614241f622f53c4eeff9890bdc4f31cfecc418b3'}


### Bước 5b — Bật 2 GPU cho Train

**Làm gì:** phiên Kaggle được cấp 2 GPU T4, nhưng Bước 5 (Test) chỉ dùng một. Cho Train, nạp 2 bản mô hình, mỗi bản trên một GPU; mỗi GPU mã hoá một nửa văn bản cùng lúc rồi ghép lại đúng thứ tự gốc.

**Kết quả thực tế:** Train chạy 45,2 phút (một GPU ước lượng khoảng 70–75 phút).

**Lưu ý khi viết bài:** Test chạy trên 1 GPU, Train chạy trên 2 GPU, cả hai đều fp16. Kết quả hai cách chạy có thể lệch cỡ làm tròn fp16 (khoảng 1e-4 ở vector) vì các batch được chia khác nhau. Số báo cáo trong bài là số trên Test; Train chỉ dùng để khai thác cặp âm khó nên không bị ảnh hưởng.

In [21]:
import ast

R = "/kaggle/working/repo/research/offline_retrieval"

# 1) Đếm và dọn bản trùng của MultiDeviceEncoder trong dense.py
path = f"{R}/dense.py"
s = open(path, encoding="utf-8").read()
marker = "class MultiDeviceEncoder:"
print("Số lần MultiDeviceEncoder xuất hiện trong dense.py:", s.count(marker))
assert s.count(marker) >= 1, "Chưa có MultiDeviceEncoder: báo lại cho tôi, không chạy tiếp"
while s.count(marker) > 1:             # các bản giống hệt nhau: bỏ bớt cho còn đúng 1
    i1 = s.index(marker)
    i2 = s.index(marker, i1 + 1)
    s = s[:i1] + s[i2:]
open(path, "w", encoding="utf-8").write(s)
ast.parse(s)                           # kiểm tra file vẫn là Python hợp lệ
print("Sau khi dọn:", s.count(marker), "bản; dense.py biên dịch được")

# 2) Vá runner.py (nếu chưa vá)
r = open(f"{R}/runner.py", encoding="utf-8").read()
if "MultiDeviceEncoder" in r:
    print("runner.py đã có MultiDeviceEncoder, bỏ qua bước vá")
else:
    patch(f"{R}/runner.py", [
    ('''    encoder = SentenceTransformerEncoder(
        model=model,
        revision=dense["revision"],
        device=dense.get("device", "cpu"),''',
    '''    devices = dense.get("devices")
    if devices is not None:
        from .dense import MultiDeviceEncoder
        if (
            not isinstance(devices, list)
            or len(devices) < 2
            or not all(isinstance(item, str) and item for item in devices)
        ):
            raise ValueError("dense.devices must be a list of at least two device names")
        encoder = MultiDeviceEncoder(
            [
                SentenceTransformerEncoder(
                    model=model,
                    revision=dense["revision"],
                    device=device,
                    batch_size=int(dense.get("batch_size", 16)),
                    query_prefix=dense.get("query_prefix", ""),
                    document_prefix=dense.get("document_prefix", ""),
                    dtype=dense.get("dtype", "float32"),
                )
                for device in devices
            ]
        )
        encoder.metadata["model"] = dense["model"]
        return encoder
    encoder = SentenceTransformerEncoder(
        model=model,
        revision=dense["revision"],
        device=dense.get("device", "cpu"),'''),
    ])
ast.parse(open(f"{R}/runner.py", encoding="utf-8").read())
print("runner.py biên dịch được")

# 3) Config Train dùng 2 GPU
import copy
train_config = copy.deepcopy(dense_config)
train_config["dense"]["devices"] = ["cuda:0", "cuda:1"]
TRAIN_CONFIG_PATH = "/kaggle/working/dense_chunking_config_train.json"
json.dump(train_config, open(TRAIN_CONFIG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Config Train dùng devices =", train_config["dense"]["devices"])

Số lần MultiDeviceEncoder xuất hiện trong dense.py: 3
Sau khi dọn: 1 bản; dense.py biên dịch được
runner.py đã có MultiDeviceEncoder, bỏ qua bước vá
runner.py biên dịch được
Config Train dùng devices = ['cuda:0', 'cuda:1']


### Bước 6 — Chạy Dense trên Train, dùng 2 GPU

**Làm gì:** chạy cùng 2 phương pháp trên pool Train (500 parent, 4.322 câu hỏi) với cấu hình 2 GPU ở Bước 5b.

**Vì sao cần Train:** kết quả Train dùng để tìm các đoạn sai khó (đoạn của tài liệu sai mà máy tìm kiếm vẫn xếp cao) cho bước fine-tune reranker. Số báo cáo trong bài là số trên Test, không phải Train.

**Kết quả đúng:** `train: rc=0`; thực tế 45,2 phút.

In [23]:
CONFIG_PATH_TEST, CONFIG_PATH = CONFIG_PATH, TRAIN_CONFIG_PATH   # run_dense đọc biến CONFIG_PATH
rc_train = run_dense(TRAIN_DIR, "train")
CONFIG_PATH = CONFIG_PATH_TEST

Đang chạy Dense: train ...
/kaggle/working/results/train/dense_chunking/run_manifest.json

train: rc=0 | 45.2 phút


### Bước 7 — Đọc kết quả Test và Train, nén zip

**Làm gì:** in Recall mức parent của hai phương pháp trên cả hai pool, thời gian từng bước thật và thiết bị đã dùng; rồi nén thư mục `results` thành `dense_chunking_results.zip` để tải về.

**Kết quả (Recall mức parent):**

| Pool | Phương pháp | R@1 | R@10 | R@100 |
|---|---|---:|---:|---:|
| Test | fixed_window_dense | 0,672 | 0,775 | 0,889 |
| Test | structure_dense | 0,608 | 0,738 | 0,854 |
| Train | fixed_window_dense | 0,674 | 0,779 | 0,873 |
| Train | structure_dense | 0,613 | 0,722 | 0,844 |

**Cách đọc:** trên cả Test và Train, cắt cửa sổ cố định tốt hơn cắt theo cấu trúc. Test chạy 1 GPU, Train chạy 2 GPU, cả hai fp16.

In [24]:
import os, json, shutil
OUT_ROOT = "/kaggle/working/results"

for split in ["test", "train"]:
    d = f"{OUT_ROOT}/{split}/dense_chunking"
    m = json.load(open(f"{d}/metrics.json"))
    print("=====", split, "=====")
    for method, v in m.items():
        if isinstance(v, dict) and "parent_macro" in v:
            row = {k: round(v["parent_macro"][k]["recall"], 4) for k in ["1", "10", "100"]}
            print(f"{method:20s} parent recall@k = {row}")
    rm = json.load(open(f"{d}/run_manifest.json"))
    print("timing_seconds:", {k: round(v, 1) for k, v in rm["timing_seconds"].items()})
    enc = rm.get("dense_encoder", {})
    print("devices:", enc.get("devices") or enc.get("device"), "| dtype:", enc.get("dtype"))

shutil.make_archive("/kaggle/working/dense_chunking_results", "zip", OUT_ROOT)
print("Đã nén:", "/kaggle/working/dense_chunking_results.zip")

===== test =====
fixed_window_dense   parent recall@k = {'1': 0.672, '10': 0.7746, '100': 0.8895}
structure_dense      parent recall@k = {'1': 0.6081, '10': 0.7378, '100': 0.8543}
timing_seconds: {'evaluation': 0.4, 'fixed_window_build': 51.6, 'fixed_window_dense': 2296.7, 'structure_dense': 1200.7}
devices: cuda | dtype: float16
===== train =====
fixed_window_dense   parent recall@k = {'1': 0.6738, '10': 0.7792, '100': 0.8729}
structure_dense      parent recall@k = {'1': 0.6131, '10': 0.7224, '100': 0.8438}
timing_seconds: {'evaluation': 0.7, 'fixed_window_build': 64.1, 'fixed_window_dense': 1680.0, 'structure_dense': 898.1}
devices: ['cuda:0', 'cuda:1'] | dtype: float16
Đã nén: /kaggle/working/dense_chunking_results.zip


### Bước 8a — Lấy nhãn loại obfuscation cho từng parent

Giống bước đã làm cho BM25: lần theo `annotation_provenance` của từng câu hỏi để tìm thẻ tương ứng trong file XML gốc của PAN-PC-11, lấy hai nhãn `type` (artificial / translation / simulated) và `obfuscation` (none / low / high). Cần dataset có thư mục `external-detection-corpus` được gắn vào notebook này.

Kết quả đúng: Test có 256 artificial / 23 translation / 21 simulated; Train có 430 / 51 / 19.

In [25]:
import glob, json, xml.etree.ElementTree as ET
from collections import Counter

ext = glob.glob('/kaggle/input/**/external-detection-corpus', recursive=True)
assert ext, "Chưa gắn dataset có external-detection-corpus vào notebook này"
INPUT_ROOT = ext[0]

def build_labels(pool_dir):
    parent_to_prov = {}
    for line in open(f"{pool_dir}/queries.jsonl", encoding="utf-8").read().split("\n"):
        if not line.strip():
            continue
        q = json.loads(line)
        parent_to_prov.setdefault(q["parent_annotation_id"], q["annotation_provenance"])
    labels = {}
    for pid, prov in parent_to_prov.items():
        root = ET.parse(f"{INPUT_ROOT}/{prov['xml_path']}").getroot()
        feats = [f for f in root.iter()
                 if f.tag.split('}')[-1] == 'feature' and f.attrib.get('name', '').casefold() == 'plagiarism']
        feat = feats[prov["feature_index"]]
        labels[pid] = {"type": feat.attrib.get("type"), "obfuscation": feat.attrib.get("obfuscation")}
    return labels

LABELS = {"test": build_labels(TEST_DIR), "train": build_labels(TRAIN_DIR)}
for s, lab in LABELS.items():
    print(s, "type:", Counter(v["type"] for v in lab.values()))

test type: Counter({'artificial': 256, 'translation': 23, 'simulated': 21})
train type: Counter({'artificial': 430, 'translation': 51, 'simulated': 19})


### Bước 8b — Dense: tách theo loại obfuscation và bootstrap cặp

**Phần 1: theo loại obfuscation.** Với mỗi phương pháp (`fixed_window_dense`, `structure_dense`) và mỗi nhóm (artificial / translation / simulated / tất cả), tính Recall trung bình **theo parent** kèm khoảng tin cậy 95% bằng bootstrap (bốc lại các parent 2.000 lần, seed 20261121 — giống hệt bên BM25 để so sánh được).

**Phần 2: bootstrap cặp.** Tính hiệu số fixed − structure trên **từng parent**, rồi bootstrap hiệu số đó. Nếu khoảng tin cậy không chứa 0 (cột "loại 0?" = có) thì chênh lệch có ý nghĩa thống kê.

**Lưu ý khi đọc:** nhóm translation (n=23 ở Test, 51 ở Train) và simulated (n=21, 19) nhỏ, nên khoảng tin cậy rộng. Đây là chỗ cần xem kỹ nhất, vì BM25 gần như mù ở translation (R@100 ≈ 0,23), và Dense đa ngôn ngữ có thể khác.

Kết quả ghi vào `paired_bootstrap.json/.md` và `by_type.json` trong thư mục `dense_chunking` của từng split, rồi nén lại thành `dense_chunking_results.zip`.

In [26]:
import numpy as np, os, json, zipfile
from collections import defaultdict

RES = "/kaggle/working/results"
KS = ("1", "10", "100")
N_BOOT, SEED = 2000, 20261121
METHODS = ("fixed_window_dense", "structure_dense")
TYPES = ("artificial", "translation", "simulated")

def load_q2p(pool_dir):
    q2p = {}
    for line in open(f"{pool_dir}/queries.jsonl", encoding="utf-8").read().split("\n"):
        if line.strip():
            q = json.loads(line)
            q2p[q["query_id"]] = q["parent_annotation_id"]
    return q2p

def parent_means(per_query, q2p):
    acc = defaultdict(lambda: {k: [] for k in KS})
    for qid, v in per_query.items():
        p = q2p.get(qid)
        if p is None:
            continue
        for k in KS:
            acc[p][k].append(v[k]["recall"])
    return {p: {k: float(np.mean(a[k])) for k in KS} for p, a in acc.items()}

def boot_ci(vals, rng):
    vals = np.asarray(vals)
    idx = rng.integers(0, len(vals), size=(N_BOOT, len(vals)))
    means = vals[idx].mean(axis=1)
    return float(vals.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

for split, pool_dir in {"test": TEST_DIR, "train": TRAIN_DIR}.items():
    labels, q2p = LABELS[split], load_q2p(pool_dir)
    d_out = f"{RES}/{split}/dense_chunking"
    m = json.load(open(f"{d_out}/metrics.json", encoding="utf-8"))
    pm = {meth: parent_means(m[meth]["per_query"], q2p) for meth in METHODS}
    common = sorted(set(pm[METHODS[0]]) & set(pm[METHODS[1]]))
    groups = {"tất cả": common}
    for t in TYPES:
        groups[t] = [p for p in common if labels[p]["type"] == t]

    rng = np.random.default_rng(SEED)
    lines = [f"# Dense trên {split}: theo loại obfuscation (bootstrap 95% theo parent, seed {SEED})", "",
             "| phương pháp | nhóm | n | R@1 | R@10 | R@100 (95% CI) |", "|---|---|---:|---:|---:|---|"]
    by_type = {}
    for meth in METHODS:
        for g, ps in groups.items():
            row = {k: boot_ci([pm[meth][p][k] for p in ps], rng) for k in KS}
            by_type[f"{meth}|{g}"] = {"n": len(ps), **{f"R@{k}": row[k] for k in KS}}
            lines.append(f"| {meth} | {g} | {len(ps)} | {row['1'][0]:.4f} | {row['10'][0]:.4f} | "
                         f"{row['100'][0]:.4f} [{row['100'][1]:.4f}, {row['100'][2]:.4f}] |")

    lines += ["", f"# Bootstrap cặp fixed_window_dense − structure_dense ({split})", "",
              "| nhóm | n | chỉ số | fixed | structure | hiệu | CI 95% | loại 0? |",
              "|---|---:|---|---:|---:|---:|---|---|"]
    paired = {}
    rng = np.random.default_rng(SEED)
    for g, ps in groups.items():
        for k in KS:
            d = [pm["fixed_window_dense"][p][k] - pm["structure_dense"][p][k] for p in ps]
            mean, lo, hi = boot_ci(d, rng)
            fx = float(np.mean([pm["fixed_window_dense"][p][k] for p in ps]))
            st = float(np.mean([pm["structure_dense"][p][k] for p in ps]))
            paired[f"{g}|R@{k}"] = {"n": len(ps), "fixed": fx, "structure": st, "diff": mean,
                                   "ci95": [lo, hi], "excludes_zero": bool(lo > 0 or hi < 0)}
            lines.append(f"| {g} | {len(ps)} | R@{k} | {fx:.4f} | {st:.4f} | {mean:+.4f} | "
                         f"[{lo:+.4f}, {hi:+.4f}] | {'có' if (lo > 0 or hi < 0) else 'không'} |")

    json.dump(by_type, open(f"{d_out}/by_type.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump(paired, open(f"{d_out}/paired_bootstrap.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    open(f"{d_out}/paired_bootstrap.md", "w", encoding="utf-8").write("\n".join(lines))
    print("\n".join(lines), "\n")

ZIP = "/kaggle/working/dense_chunking_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RES):
        for fn in files:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, "/kaggle/working"))
print("Đã ghi:", ZIP, f"({os.path.getsize(ZIP)/1e6:.1f} MB)")

# Dense trên test: theo loại obfuscation (bootstrap 95% theo parent, seed 20261121)

| phương pháp | nhóm | n | R@1 | R@10 | R@100 (95% CI) |
|---|---|---:|---:|---:|---|
| fixed_window_dense | tất cả | 300 | 0.6720 | 0.7746 | 0.8895 [0.8601, 0.9168] |
| fixed_window_dense | artificial | 256 | 0.6607 | 0.7671 | 0.8957 [0.8668, 0.9229] |
| fixed_window_dense | translation | 23 | 0.5642 | 0.6520 | 0.7193 [0.5415, 0.8936] |
| fixed_window_dense | simulated | 21 | 0.9286 | 1.0000 | 1.0000 [1.0000, 1.0000] |
| structure_dense | tất cả | 300 | 0.6081 | 0.7378 | 0.8543 [0.8191, 0.8854] |
| structure_dense | artificial | 256 | 0.5935 | 0.7257 | 0.8558 [0.8214, 0.8889] |
| structure_dense | translation | 23 | 0.4561 | 0.6324 | 0.7043 [0.5162, 0.8751] |
| structure_dense | simulated | 21 | 0.9524 | 1.0000 | 1.0000 [1.0000, 1.0000] |

# Bootstrap cặp fixed_window_dense − structure_dense (test)

| nhóm | n | chỉ số | fixed | structure | hiệu | CI 95% | loại 0? |
|---|---:|---|---:|---:|---:|---|--

## Phụ lục — Xuất đáp án và nhãn obfuscation để phân tích ngoài Kaggle

Zip kết quả chỉ chứa điểm theo câu hỏi, không chứa tài liệu nguồn đúng và loại obfuscation của từng parent. Ô này xuất file `meta_for_analysis.json` (nhỏ) gồm, cho Test và Train: với mỗi câu hỏi, mã parent và danh sách tài liệu nguồn đúng; với mỗi parent, `type` và `obfuscation`. Dùng cho bootstrap cặp Dense vs BM25 và cho RRF, chạy ở ngoài Kaggle bằng CPU.

In [27]:
import glob, json, os, shutil, xml.etree.ElementTree as ET

ext = glob.glob('/kaggle/input/**/external-detection-corpus', recursive=True)
assert ext, "Chưa gắn dataset có external-detection-corpus"
INPUT_ROOT = ext[0]

def pool_dir(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f"Không thấy {name}: gắn dataset pool vào notebook"
    return hits[0]

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8").read().split("\n") if l.strip()]

def export(pool):
    queries = read_jsonl(f"{pool}/queries.jsonl")
    qrels = read_jsonl(f"{pool}/qrels.jsonl")
    parent_prov, q2p = {}, {}
    for q in queries:
        q2p[q["query_id"]] = q["parent_annotation_id"]
        parent_prov.setdefault(q["parent_annotation_id"], q["annotation_provenance"])
    gold = {}
    for r in qrels:
        gold.setdefault(r["query_id"], set()).add(r["document_id"])
    labels = {}
    for pid, prov in parent_prov.items():
        root = ET.parse(f"{INPUT_ROOT}/{prov['xml_path']}").getroot()
        feats = [f for f in root.iter()
                 if f.tag.split('}')[-1] == 'feature' and f.attrib.get('name', '').casefold() == 'plagiarism']
        feat = feats[prov["feature_index"]]
        labels[pid] = {"type": feat.attrib.get("type"), "obfuscation": feat.attrib.get("obfuscation")}
    return {
        "query_to_parent": q2p,
        "query_gold_docs": {q: sorted(d) for q, d in gold.items()},
        "parent_labels": labels,
    }

meta = {"test": export(pool_dir("pan-pc11-test-v1")),
        "train": export(pool_dir("pan-pc11-train-v1"))}
out = "/kaggle/working/meta_for_analysis.json"
json.dump(meta, open(out, "w", encoding="utf-8"), ensure_ascii=False)
for s, m in meta.items():
    print(s, "| câu hỏi:", len(m["query_to_parent"]), "| có đáp án:", len(m["query_gold_docs"]),
          "| parent:", len(m["parent_labels"]))
print("Đã ghi:", out, f"({os.path.getsize(out)/1e3:.0f} KB)")

test | câu hỏi: 2606 | có đáp án: 2606 | parent: 300
train | câu hỏi: 4322 | có đáp án: 4322 | parent: 500
Đã ghi: /kaggle/working/meta_for_analysis.json (1045 KB)
